# Data Consistency and Standardization

## Objective

In this notebook, we will cover:

### Text Cleaning
- Convert to Lowercase
- Remove Extra Spaces
- Remove Special Characters
- Unicode Normalization
- Spelling Correction

### Numerical Scaling
- Min-Max Scaling
- Standard Scaling
- Robust Scaling
- Log Transformation
- Power Transformation

The original dataset will remain unchanged.

In [ ]:
import numpy as np
import pandas as pd
import unicodedata

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    PowerTransformer
)

In [ ]:
df = pd.read_csv(
    "../datasets/heart_failure_clinical_records_dataset-selected-columns.csv"
)

df.head()

In [ ]:
# Create a copy for practice
df_clean = df.copy()

print("Dataset Shape:", df_clean.shape)
print("Original Dataset Preserved:", df.equals(df_clean))

# Text Cleaning

The Heart Failure dataset does not contain suitable text columns for demonstrating all text-cleaning techniques.

Therefore, a small practice text column will be created without modifying the original dataset.

In [ ]:
practice_text = pd.DataFrame({
    "city": [
        " LAHORE ",
        "lahore",
        "Lahor!",
        " KARACHI",
        "karachi#",
        "Islamabad ",
        "ISLAMABAD"
    ]
})

practice_text

In [ ]:
practice_text["city_lower"] = (
    practice_text["city"]
    .str.lower()
)

practice_text[
    ["city", "city_lower"]
]

In [ ]:
practice_text["city_no_spaces"] = (
    practice_text["city_lower"]
    .str.strip()
)

practice_text[
    ["city_lower", "city_no_spaces"]
]

In [ ]:
practice_text["city_clean"] = (
    practice_text["city_no_spaces"]
    .str.replace(
        r"[^a-zA-Z0-9\s]",
        "",
        regex=True
    )
)

practice_text[
    ["city_no_spaces", "city_clean"]
]

## Unicode Normalization

Unicode Normalization converts different Unicode representations into a consistent standard form.

In [ ]:
practice_text["city_unicode"] = (
    practice_text["city_clean"]
    .apply(
        lambda text: unicodedata.normalize(
            "NFKC",
            text
        )
    )
)

practice_text[
    ["city_clean", "city_unicode"]
]

## Spelling Correction

Known spelling inconsistencies can be corrected using a validated mapping.

Automatic spelling correction should not be applied blindly because similar words may represent different valid categories.

In [ ]:
spelling_corrections = {
    "lahor": "lahore"
}

practice_text["city_final"] = (
    practice_text["city_unicode"]
    .replace(spelling_corrections)
)

practice_text[
    ["city", "city_final"]
]

In [ ]:
print("Original Categories:")
print(practice_text["city"].unique())

print("\nCleaned Categories:")
print(practice_text["city_final"].unique())

# Numerical Scaling

Numerical features can have very different measurement scales.

For this dataset, we will demonstrate scaling using continuous numerical features.

Binary categorical columns such as `diabetes`, `sex`, `anaemia`, and `high_blood_pressure` will not be included.

In [ ]:
numerical_features = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_creatinine",
    "serum_sodium"
]

df[numerical_features].describe()

## Min-Max Scaling

Min-Max Scaling transforms numerical values to a fixed range, usually between 0 and 1.

In [ ]:
minmax_scaler = MinMaxScaler()

minmax_values = minmax_scaler.fit_transform(
    df[numerical_features]
)

df_minmax = pd.DataFrame(
    minmax_values,
    columns=numerical_features
)

df_minmax.head()

In [ ]:
print("Minimum Values:")
print(df_minmax.min())

print("\nMaximum Values:")
print(df_minmax.max())

## Standard Scaling

Standard Scaling centers features around zero using their mean and standard deviation.

In [ ]:
standard_scaler = StandardScaler()

standard_values = standard_scaler.fit_transform(
    df[numerical_features]
)

df_standard = pd.DataFrame(
    standard_values,
    columns=numerical_features
)

df_standard.head()

In [ ]:
print("Means:")
print(
    df_standard.mean().round(2)
)

print("\nStandard Deviations:")
print(
    df_standard.std(ddof=0).round(2)
)

## Robust Scaling

Robust Scaling uses the median and IQR, making it less sensitive to extreme values.

In [ ]:
robust_scaler = RobustScaler()

robust_values = robust_scaler.fit_transform(
    df[numerical_features]
)

df_robust = pd.DataFrame(
    robust_values,
    columns=numerical_features
)

df_robust.head()

In [ ]:
scaling_comparison = pd.DataFrame({
    "Original": df["serum_creatinine"],
    "MinMax": df_minmax["serum_creatinine"],
    "Standard": df_standard["serum_creatinine"],
    "Robust": df_robust["serum_creatinine"]
})

scaling_comparison.head(10)

## Log Transformation

Log Transformation can reduce right-skewness and compress large positive values.

It is not the same as ordinary feature scaling.

In [ ]:
log_feature = "creatinine_phosphokinase"

df_log = df.copy()

df_log[
    "creatinine_phosphokinase_log"
] = np.log1p(
    df_log[log_feature]
)

df_log[
    [
        log_feature,
        "creatinine_phosphokinase_log"
    ]
].head(10)

In [ ]:
original_skew = df[
    log_feature
].skew()

log_skew = df_log[
    "creatinine_phosphokinase_log"
].skew()

print(
    "Original Skewness:",
    round(original_skew, 2)
)

print(
    "After Log Transformation:",
    round(log_skew, 2)
)

## Power Transformation

Power Transformation attempts to make numerical distributions more symmetric.

We will use the Yeo-Johnson method.